In [44]:
import pandas as pd

df = pd.read_csv("C:/xampp/htdocs/mini project/insurance_claims_dataset.csv")

df.head()

,claim_id,age,vehicle_age,claim_amount,accident_type,previous_claims,police_report,witness_present,garage_type,days_to_claim,fraud
0,CLM1000,41,14,179725,Fire,4,0,1,Local,54,0
1,CLM1001,58,13,336738,Natural Disaster,4,1,0,Authorized,46,1
2,CLM1002,65,13,114476,Collision,0,0,1,Local,35,0
3,CLM1003,26,1,50833,Collision,0,0,0,Local,25,1
4,CLM1004,68,9,212820,Theft,1,1,0,Authorized,21,0


In [37]:
# Rule-based fraud feature
df["rule_fraud"] = 0

df.loc[(df["claim_amount"] > 200000) & (df["police_report"] == 0), "rule_fraud"] = 1
df.loc[df["previous_claims"] > 3, "rule_fraud"] = 1
df.loc[df["days_to_claim"] > 30, "rule_fraud"] = 1

# Extra features
df["high_amount"] = (df["claim_amount"] > 150000).astype(int)
df["late_claim"] = (df["days_to_claim"] > 15).astype(int)
df["many_claims"] = (df["previous_claims"] > 2).astype(int)
df["no_police"] = (df["police_report"] == 0).astype(int)

In [38]:
from sklearn.utils import resample

df_majority = df[df.fraud == 0]
df_minority = df[df.fraud == 1]

df_minority_upsampled = resample(df_minority,
                                replace=True,
                                n_samples=len(df_majority),
                                random_state=42)

df = pd.concat([df_majority, df_minority_upsampled])

In [39]:
X = df.drop(["claim_id","fraud"], axis=1)
y = df["fraud"]

X = pd.get_dummies(X)

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [47]:
from sklearn.metrics import accuracy_score, classification_report

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.9076305220883534
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       257
           1       0.90      0.90      0.90       241

    accuracy                           0.91       498
   macro avg       0.91      0.91      0.91       498
weighted avg       0.91      0.91      0.91       498



In [1]:
import os
print(os.getcwd())

C:\Users\Admin
